# bridge_agents

> Phase 20 (`specs/multiagent/phase20/`, experiment C1): the agents' side of the bridge board. The rule sentences (`COUPLE_RULE`, `MIRROR_RULE`, `BRIDGE_RULE`, which open with the one exception to the unchanged squad rule) and `bridgeRules`; `bridgeLines`, at most three observation lines (a holder's mirrors, a bridge's pair, the squad's coupled subtasks); `parseBridge` (mirror shares, which never reach the channel, and relays); the numpy members `RuleBridge` (relays, and honest mirror posts, P22-12) and `MirrorLiar` (scripted liars that also lie on mirrors), applied by `bridgeMembers` through `colony_board.stage4Members`; and `bridgePromptTokens`, the longest prompt per role in a rollout. Without a bridge key nothing changes.

In [ ]:
#| default_exp bridge_agents

In [ ]:
#| hide
from nbdev.showdoc import *

## Rules, lines and parsing

In [ ]:
#| export
import re
from functools import lru_cache

from explRL.bridge_board import BRIDGE_KEYS, BridgeBoardEnv, BridgeView
from explRL.clue_agents import ClueLiar, RuleAgent
from explRL.clue_colony import Q, leading
from explRL.colony_agents import MistakenAgent
from explRL.colony_board import InSquad

COUPLE_RULE = ("About a quarter of your squad's subtasks need one clue held in squad {s2}; they are listed under 'Your subtasks that need squad {s2}'. "
               "Nobody in your squad holds that clue.")
MIRROR_RULE = ("This is the one exception to the squad rule: some subtasks of squad {s0} need a clue that you or your squad hold; they are listed under 'For squad {s0}'. "
               "Share your clue there. You cannot open or answer them.")
BRIDGE_RULE = ("This is the one exception to the squad rule: you are the bridge from squad {s2} to your squad: you also see what squad {s2} posted for your squad's "
               "subtasks that need it. Relay a value by sharing it on the matching subtask of your squad.")
MIRROR_HEAD, PAIR_HEAD, NEED_HEAD = 'For squad {s0} (share your clue; you cannot answer): ', 'From squad {s2} for your subtasks: ', \
    'Your subtasks that need squad {s2}: '
_CLUE = re.compile(r'x\s*mod\s*(\d+)\s*=\s*(-?\d+)', re.I)          # clue_text.CLUE_RE (clue_text imports this module)


def bridgeRules(env, i):
    """The bridge sentences for agent i, each after a blank line ('' on any other board, or without i): COUPLE_RULE and
    MIRROR_RULE on a coupled board (every squad holds cross clues for the one before it, so any agent may hold one),
    BRIDGE_RULE for a bridge (a guarding bridge keeps GUARD_RULE before it). SQUAD_RULE is unchanged (P20-15)."""
    b = getattr(env, 'world', None)
    if i is None or not isinstance(b, BridgeBoardEnv) or not b.couple:
        return ''
    s0, s2 = (i - 1) % b.nSquads, (i + 1) % b.nSquads
    out = [COUPLE_RULE.format(s2=s2), MIRROR_RULE.format(s0=s0)] + ([BRIDGE_RULE.format(s2=s2)] if b.bridgeOf(i) else [])
    return ''.join('\n\n' + x for x in out)


def _vals(p, vs):
    return ', '.join(f"x mod {p} = {r} ({len(s)} agent{'s' * (len(s) != 1)})" for r, s, _ in vs) or 'none'


def bridgeLines(view, i, t=None):
    """At most three observation lines for agent i on a bridge board ([] on any other): its mirrors, for holders of a cross
    clue (P20-16; liar slots: all), with the clue it holds and the values posted there; for a bridge, its pair's mirror
    posts (P20-3); for every agent, its squad's open subtasks that need the next squad."""
    if not isinstance(view, BridgeView):
        return []
    b, s0, s2, out = view._b, (i - 1) % view.nSquads, (i + 1) % view.nSquads, []
    if mine := view.mirrors():
        held = view.crossClues()
        out.append(MIRROR_HEAD.format(s0=s0) + '; '.join(
            f"T{k}: x mod {b.tasks[k].task.primes[1]}{f' = {held[k][1]} (yours)' if k in held else ''}; posted: "
            + _vals(b.tasks[k].task.primes[1], b.mirrorTally(k).get(k, {}).get(1, [])) for k in mine))
    if (pair := view.pair) is not None:
        out.append(PAIR_HEAD.format(s2=s2) + ('; '.join(f'T{k}: ' + _vals(b.tasks[k].task.primes[1], pair[k].get(1, []))
                                                         for k in view.open if k in pair) or 'none'))
    if need := [k for k in view.open if b.coupled.get(k, (None,))[0] == i % b.nSquads]:
        out.append(NEED_HEAD.format(s2=s2) + ', '.join(f'T{k}' for k in need))
    return out


def parseBridge(env, parsed, i):
    """TextClueEnv.parse's bridge cases -> (action, report, valid), or None for parse's own path. A share on one of i's
    mirrors: valid at the cross clue's true residue from a holder (an adaptive liar: any residue), with report None, so
    nothing reaches the channel; anything else there is invalid. A bridge's share on its pair's open coupled subtask of a
    value posted on the mirror (any value, P20-4): a relay, valid and posted as an ordinary share."""
    b = env.world
    if not isinstance(b, BridgeBoardEnv) or not isinstance(parsed, dict) or parsed.get('action') != 'share':
        return None
    k, m = env._task(parsed.get('target', '')), _CLUE.search(str(parsed.get('clue', '')))
    if k not in b.coupled or k not in b.open or m is None:
        return None
    (p, r), (p1, r1) = (int(m.group(1)), int(m.group(2))), b.tasks[k].task.clue(1)
    if k in b.mirrors(i):
        liar = env.fault == 'adaptive' and i < env.nLiar
        ok = p == p1 and (r == r1 and i in b.tasks[k].holders[1] or liar and 0 <= r < p)
        return (('share', (k, 1, r)), None, True) if ok else (('wait', None), None, False)
    if b.bridgeOf(i) == b.coupled[k] and p == p1 and any(x[2:5] == (k, 1, r) for x in b.mirrorLog):
        return ('share', (k, 1, r)), (k, (1, p, r)), True
    return None

## Rule agents, liars and prompt length

In [ ]:
#| export
def onMirror(b, i, action):
    "Whether `action` is a share on another squad's coupled subtask, i.e. on one of agent i's mirrors (never posted to the channel)."
    c = action[0] == 'share' and getattr(b, 'coupled', {}).get(int(action[1][0]))
    return bool(c) and c[1] == i % b.nSquads


class RuleBridge(RuleAgent):
    """Mixin for an honest numpy member on a bridge board (`bridgeMembers` builds `Bridge<class>` from its class, so knownFor
    and everything else stay the base agent's: a ColonyAgent on colony arms, flag-aware on failFlag boards). Roles are read at
    act time. A bridge (board.bridgeOf) first relays the leading mirror value (most senders, then earliest) of an open coupled
    subtask of its pair with no value of the cross clue on its board; otherwise it acts as its base agent. Every member, at
    its sharing step, first posts a held cross clue on its mirror when the mirror's leading value is missing, different or
    unconfirmed (ColonyAgent's rule, P22-12). A mirror share is applied to the board only, never posted to the channel."""
    def choose(self, b, t):
        return self._relay(b) or super().choose(b, t)

    def _relay(self, b):
        p = b.bridgeOf(self.id) if hasattr(b, 'bridgeOf') else None
        if p is None or self.channel is None:
            return None
        for k in b.open:
            if b.coupled.get(k) == p and not any(s[2] == k and s[3] == 1 for s in b.shareLog):
                first = {}
                for x in b.mirrorLog:
                    if x[2] == k and x[3] == 1:
                        first.setdefault(x[4], (x[0], set()))[1].add(x[1])
                if first:
                    return ('share', (k, 1, int(min(first, key=lambda r: (-len(first[r][1]), first[r][0])))))
        return None

    def _shareable(self, b, kn, t, *a):
        return self._mirrorShare(b) or super()._shareable(b, kn, t, *a)

    def _mirrorShare(self, b):
        if self.channel is None or self.share == 'never' or not hasattr(b, 'mirrors'):
            return None
        q, held = getattr(self, 'q', Q), getattr(self, 'heldFor', lambda b, k: b.held(self.id, k))
        for k in b.mirrors(self.id, False):
            r, lead = held(b, k).get(1), leading(b.mirrorTally(k), k, 1)
            if r is not None and (lead is None or lead[0] != r or len(lead[1]) < q and self.id not in lead[1]):
                return ('share', (k, 1, r))
        return None

    def post(self, b, action, t):
        if not onMirror(b, self.id, action):
            super().post(b, action, t)


class MirrorLiar(ClueLiar):
    """ClueLiar that counts its squad's mirrors (all of them, P20-5) among its targets: on a lying turn it draws k from
    open + mirrors with rng.integers as ClueLiar does, and on a mirror posts a false residue of clue 1 (to the board only).
    With no mirrors (no bridge key, or none open) its draws are ClueLiar's."""
    def choose(self, b, t):
        mir = b.mirrors(self.id, True) if hasattr(b, 'mirrors') else []
        if not mir:
            return super().choose(b, t)
        self.turn += 1
        cand = [k for k in b.open if k != b.opened[self.id]] + mir
        if self.channel is not None and self.turn % 2:
            k = int(cand[int(self.rng.integers(len(cand)))])
            task = b.tasks[k].task
            j = 1 if k in mir else int(self.rng.integers(len(task.primes)))
            p, r = task.clue(j)
            return ('share', (k, j, int((r + 1 + self.rng.integers(p - 1)) % p)))
        return RuleAgent.choose(self, b, t)

    def post(self, b, action, t):
        if not onMirror(b, self.id, action):
            super().post(b, action, t)


@lru_cache(None)
def _bridged(cls):
    return type('Bridge' + cls.__name__, (RuleBridge, cls), {})


def bridgeMembers(world, members):
    """A run's numpy members on a bridge board (stage4Members' hook, so every numpy path gets it): scripted ClueLiars become
    MirrorLiar; honest rule agents (not liars or mistaken members) gain RuleBridge (their class becomes Bridge<class>).
    InSquad wrappers are kept (their inner member changes); Claude members are left alone. Returns `members`."""
    for m in members:
        x = m.m if isinstance(m, InSquad) else m
        if type(x) is ClueLiar:
            x.__class__ = MirrorLiar
        elif isinstance(x, RuleAgent) and not isinstance(x, (ClueLiar, MistakenAgent, RuleBridge)):
            x.__class__ = _bridged(type(x))
    return members


def bridgePromptTokens(cfg, seeds, tok=None, policy=None):
    """{'bridge', 'holder', 'other': n}: the longest rendered prompt per role over a rollout of cfg on `seeds` (scripted faults
    as runEval; policy default: the fake Claude client). Role per turn: bridge slots; holders, turns that show a mirror line;
    the rest. tok: a tokenizer (chat-template text, as LocalPolicy) or text -> count; default about 4 characters a token."""
    from explRL.clue_policy import claudePolicy, userText
    from explRL.clue_roll import rollout
    from explRL.colony_agents import scriptedFaults
    from explRL.llm_client import LLMClient
    from explRL.llm_fake import FakeClient, fakeReply
    from explRL.local_policy import promptText
    n = ((lambda s, bl: len(tok(promptText(tok, s, bl), add_special_tokens=False)['input_ids'])) if hasattr(tok, 'apply_chat_template')
         else lambda s, bl: (tok or (lambda x: len(x) // 4))(s + '\n\n' + userText(bl)))
    policy = policy or claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
    ro, out = rollout(cfg, seeds, policy, scriptedFaults(cfg.world, cfg.nLiar), keepPrompts=True), dict(bridge=0, holder=0, other=0)
    head = MIRROR_HEAD.split('{')[0]
    for bd, ps, lg in zip(ro.boards, ro.prompts, ro.logs):
        for i, (pi, li) in enumerate(zip(ps, lg)):
            for (s, bl), x in zip(pi, li):
                role = 'bridge' if getattr(bd, 'bridgeOf', lambda i: None)(i) else 'holder' if f'\n{head}' in x['obs'] else 'other'
                out[role] = max(out[role], n(s, bl))
    return out

## Tests

In [ ]:
# test_relay (AC-4, AC-3's prompt half for a Claude poster, FR-B2, P20-4): through TextClueEnv.parse, a holder's mirror share is
# valid with no report, so nothing reaches the channel or squad s's prompts; a bridge's relay of a minority mirror value (a
# liar's) is valid, posted and logged with its source; a non-bridge's share of a value it does not know is invalid, and so are
# mirror shares by non-holders, at a wrong value or prime, and opening a mirror
from explRL.bridge_board import relaySource
from explRL.clue_agents import ClueBoard
from explRL.clue_colony import knownColony, tally
from explRL.clue_text import TextClueEnv
from explRL.colony_board import squadView
from explRL.colony_grid import WORLD18

kw = dict(WORLD18)
b = BridgeBoardEnv(1800, 16, guardSlots=range(4, 8), squads=4, nLiar=4, couple=0.25, bridges=1, **kw)
env, ch = TextClueEnv(b, colony=True, guards=4, nLiar=4, squads=4), ClueBoard()
k = next(k for k in b.open if k in b.coupled and min(b.tasks[k].holders[1]) >= 4)     # both cross-clue holders honest
s, s2 = b.coupled[k]
(p0, r0), (p1, r1) = b.tasks[k].task.clue(0), b.tasks[k].task.clue(1)
h1, h2 = sorted(b.tasks[k].holders[1])
br, liar = b.bridges[s][0], next(i for i in range(4) if i % 4 == s2)
other = next(i for i in range(4, 16) if i % 4 == s and i != br)                       # squad s, not the bridge
non = next((i for i in range(4, 16) if i % 4 == s2 and i not in (h1, h2)), None)       # squad s2, no cross clue
lie = (r1 + 1) % p1
share = lambda k, p, r: dict(action='share', target=f'T{k}', clue=f'x mod {p} = {r}', answer=0, notes='')

def turn(i, parsed, t, e=env):
    "One Claude turn as LLMMember.commit: observe, parse, apply, post."
    obs = e.observe(i, t, ch)
    a, rep, ok = e.parse(parsed, i)
    out = b.apply(i, a, t)
    e.post(i, rep, t, ch, out[3])
    return obs, a, rep, ok, out

obs, a, rep, ok, out = turn(h1, share(k, p1, r1), 1)
assert f'T{k}: x mod {p1} = {r1} (yours); posted: none' in obs and (a, rep, ok) == (('share', (k, 1, r1)), None, True)
assert out[3]['result'] == 'shared' and b.mirrorLog[-1][1:5] == (h1, k, 1, r1) and not ch.posts.get(k) and not b.shareLog
assert env.summary(h1, 1, a, out[0], rep, out[3]) == f'turn 1: shared T{k}: x mod {p1} = {r1}, earned {out[0]:.2f}.'
turn(h2, share(k, p1, r1), 2)
b.apply(liar, ('share', (k, 1, lie)), 2)                                                # a scripted liar's mirror post
assert [m[1] for m in b.mirrorLog] == [h1, h2, liar] and not ch.posts.get(k)
# AC-3: squad s (not its bridge) never sees the mirror posts: tally, knownColony, the board line, the whole prompt
o = env.observe(other, 3, ch)
assert k not in tally(ch, squadView(b, other)) and 1 not in knownColony(other, squadView(b, other), ch)[k]
assert f'T{k}: x mod {p1}' not in o and f'Your subtasks that need squad {s2}: ' in o and f'T{k}' in o.split('need squad')[1]
for x, why in ((share(k, p1, lie), 'unknown'), (share(k, p1, r1), 'unknown')):            # AC-4: a non-bridge cannot post them
    assert env.parse(x, other) == (('wait', None), None, False), why
# the bridge sees its pair's mirror posts (P20-3) and relays the liar's minority value (P20-4)
obs, a, rep, ok, out = turn(br, share(k, p1, lie), 3)
assert f'From squad {s2} for your subtasks: T{k}: x mod {p1} = {r1} (2 agents), x mod {p1} = {lie} (1 agent)' in obs
assert (a, rep, ok) == (('share', (k, 1, lie)), (k, (1, p1, lie)), True) and ch.posts[k][-1][:4] == (1, p1, lie, br)
assert b.relayLog[-1][1:5] == (br, k, 1, lie) and b.relayLog[-1][5] == b.mirrorLog[2] and relaySource(b, b.shareLog[-1])[1] == liar
assert turn(br, share(k, p1, r1), 4)[3] and b.relayLog[-1][5] == b.mirrorLog[0]        # and the majority value, from h1's post
assert env.parse(share(k, p1, (r1 + 2) % p1 if (r1 + 2) % p1 != lie else (r1 + 3) % p1), br)[2] is False   # never posted
# invalid mirror moves: a wrong value, the wrong prime, opening it, a non-holder of s2, another squad
bad = [(h1, share(k, p1, lie)), (h1, share(k, p0, r0)), (h1, dict(action='open', target=f'T{k}')),
       (next(i for i in range(4, 16) if i % 4 not in (s, s2)), share(k, p1, r1))] + ([(non, share(k, p1, r1))] if non else [])
for i, x in bad:
    env.observe(i, 5, ch)
    assert env.parse(x, i) == (('wait', None), None, False), (i, x)
# an adaptive (Claude) liar of s2 may post any residue on a mirror, still with no report (FR-B5)
envA = TextClueEnv(b, colony=True, guards=4, nLiar=4, squads=4, fault='adaptive')
envA.observe(liar, 5, ch)
assert envA.parse(share(k, p1, lie), liar) == (('share', (k, 1, lie)), None, True)
assert env.parse(share(k, p1, lie), liar)[2] is False                                    # not when the liars are scripted

In [ ]:
# test_bridge_liars (AC-5, AC-3's numpy half, FR-A4, FR-B4, P20-5, P22-12): MirrorLiar's draws equal ClueLiar's without mirrors;
# with them it lies on mirrors in proportion to their share of open + mirrors, with false cross clues; honest rule holders post
# true cross clues on mirrors by ColonyAgent's rule; RuleBridge relays the leading mirror value (true when the mirror leads with
# it); bridgeMembers keeps InSquad and the base classes (flag-aware knownFor); on a 64-agent numpy board no mirror post reaches
# the channel, and both liars' and honest values reach the mirrors and the relay log
import numpy as np
from explRL.bridge_agents import MirrorLiar, RuleBridge    # the module's classes, which stage4Members' hook applies
from explRL.clue_agents import ClueLiar
from explRL.clue_runs import clueSetup
from explRL.colony_agents import ColonyAgent, ColludingLiar, GuardAgent, memberClass
from explRL.colony_board import ColonyBoardEnv, InSquad, _flagKnown, stage4Members
from explRL.llm_agent import runTurns
from explRL.llm_runs import LLMConfig
from explRL.multi import spawnRngs

def play(board, cls, slots=(0, 1, 2), T=30):
    "Liars at `slots`, acting alone in turn order on their squad's view; their actions."
    ms, out = [cls(ClueBoard(), np.random.default_rng(7 + i)) for i in slots], []
    for m, i in zip(ms, slots):
        m.id = i
    for t in range(1, T):
        for m in ms:
            v = squadView(board, m.id)
            v.advance(t)
            a = m.choose(v, t)
            board.apply(m.id, a, t)
            out.append(a)
    return out

G16 = dict(guardSlots=range(4, 8), squads=4, **kw)
ref = play(ColonyBoardEnv(1800, 16, **G16), ClueLiar)
assert play(ColonyBoardEnv(1800, 16, **G16), MirrorLiar) == ref                            # no bridge key: ClueLiar's draws
assert play(BridgeBoardEnv(1800, 16, nLiar=4, couple=0, **G16), MirrorLiar) == ref         # a bridge board without mirrors
lies = play(BridgeBoardEnv(1800, 16, nLiar=4, couple=0.25, bridges=1, **G16), MirrorLiar)
assert lies != ref and any(a[0] == 'share' and a[1][1] == 1 for a in lies)
# in proportion: one lying turn each for 4,000 liars of s2 on the same board
b = BridgeBoardEnv(1800, 64, guardSlots=range(16, 32), squads=8, nLiar=16, couple=0.25, bridges=1, **kw)
liar = next(i for i in range(16) if b.mirrors(i))
v = squadView(b, liar)
mir, cand = v.mirrors(), [k for k in v.open if k != b.opened[liar]] + v.mirrors()
acts = []
for n in range(4000):
    m = MirrorLiar(ClueBoard(), np.random.default_rng(n))
    m.id = liar
    acts.append(m.choose(v, 0))
on = [a for a in acts if a[1][0] in mir]
assert abs(len(on) / len(acts) - len(mir) / len(cand)) < 0.025 and all(a[0] == 'share' for a in acts)
assert all(a[1][1] == 1 and a[1][2] != b.tasks[a[1][0]].task.clue(1)[1] for a in on)       # false residues of the cross clue
assert all(b.tasks[a[1][0]].task.clue(a[1][1])[1] != a[1][2] for a in acts)                 # every lie is false
# honest holders post true cross clues on mirrors (P22-12), RuleBridge relays the leading value, mirror shares skip the channel
cfg = LLMConfig(task='clue', channel='blackboard', N=16, turns=10, nLiar=4, world=dict(kw, colony=True, guards=4, squads=4, couple=0.25, bridges=1))
b, ch = clueSetup(cfg, 1800)
assert isinstance(b, BridgeBoardEnv) and b.bridgeSet == {8, 9, 10, 11}
mk = lambda i: memberClass(cfg.world, i, 4)(ch, np.random.default_rng(i)) if i >= 4 else ClueLiar(ch, np.random.default_rng(i))
ms = stage4Members(cfg.world, [mk(i) for i in range(16)])
for i, m in enumerate(ms):
    m.id = i
assert all(isinstance(m, InSquad) for m in ms) and all(type(m.m) is MirrorLiar for m in ms[:4])
assert type(ms[4].m).__name__ == 'BridgeGuardAgent' and isinstance(ms[4].m, GuardAgent) and isinstance(ms[9].m, (RuleBridge, ColonyAgent))
k = next(k for k in b.open if k in b.coupled and min(b.tasks[k].holders[1]) >= 4)
s, s2 = b.coupled[k]
h1, h2 = sorted(b.tasks[k].holders[1])
p1, r1 = b.tasks[k].task.clue(1)
br = ms[b.bridges[s][0]]
lie = next(m for m in ms[:4] if m.id % 4 == s2)
assert ms[h1].m._mirrorShare(squadView(b, h1)) == ('share', (k, 1, r1))                    # the mirror shows nothing yet
ms[h1].m.post(squadView(b, h1), ('share', (k, 1, r1)), 0)
assert not ch.posts                                                                         # mirror shares skip the channel
assert br.m._relay(squadView(b, br.id)) is None                                             # nothing posted on the mirror yet
b.apply(lie.id, ('share', (k, 1, (r1 + 1) % p1)), 1)                                        # the liar posts first ...
assert br.m._relay(squadView(b, br.id)) == ('share', (k, 1, (r1 + 1) % p1))                 # ... and leads: a relayed lie
b.apply(h1, ('share', (k, 1, r1)), 2)
assert br.m._relay(squadView(b, br.id)) == ('share', (k, 1, (r1 + 1) % p1))                 # a tie on senders: the earliest
assert ms[h1].m._mirrorShare(squadView(b, h1)) is None                                      # its own, unconfirmed: no repost
assert ms[h2].m._mirrorShare(squadView(b, h2)) == ('share', (k, 1, r1))                     # unconfirmed: confirm it
b.apply(h2, ('share', (k, 1, r1)), 3)
assert br.m._relay(squadView(b, br.id)) == ('share', (k, 1, r1))                            # the true value leads (2 senders)
assert all(m.m._mirrorShare(squadView(b, m.id)) is None or m.m._mirrorShare(squadView(b, m.id))[1][0] != k for m in ms[4:])
b.apply(br.id, ('share', (k, 1, r1)), 4)
assert br.m._relay(squadView(b, br.id)) is None or br.m._relay(squadView(b, br.id))[1][0] != k   # its board has the clue now
# failFlag: the bridge keeps the flag-aware knownFor of its base agent; ColludingLiar and Claude members are left alone
fw = dict(cfg.world, failFlag=True)
fm = stage4Members(fw, [ColludingLiar(ch, np.random.default_rng(0))] + [memberClass(fw, i, 1)(ch, np.random.default_rng(i)) for i in (1, 9)] + ['claude'])
assert type(fm[0].m) is ColludingLiar and fm[3] == 'claude' and fm[2].m.knownFor.__func__ is _flagKnown
assert type(fm[2].m).__name__ == 'BridgeFlagColonyAgent' and type(fm[1].m).__name__ == 'BridgeFlagGuardAgent'
assert stage4Members(fw, fm)[2].m.__class__ is fm[2].m.__class__                            # applied once
# a 64-agent numpy board (bridge1 at seed 1890): mirrors get honest and false posts, relays carry both, the channel none
cfg = LLMConfig(task='clue', channel='blackboard', N=64, turns=40, nLiar=16,
                world=dict(kw, colony=True, guards=16, squads=8, couple=0.25, bridges=1))
b, ch = clueSetup(cfg, 1890)
rngs = spawnRngs(1890 + 1000, 64)
runTurns(stage4Members(cfg.world, [ClueLiar(ch, rngs[i]) if i < 16 else memberClass(cfg.world, i, 16)(ch, rngs[i]) for i in range(64)]), b, 40)
mp = b.mirrorLog
assert sum(m[1] >= 16 and not m[6] for m in mp) > 50 and sum(m[1] < 16 and m[6] for m in mp) > 10
assert not any(m[1] >= 16 and m[6] for m in mp) and not any(m[1] < 16 and not m[6] for m in mp)
assert sum(not r[5][6] for r in b.relayLog) > 20 and sum(r[5][6] for r in b.relayLog) > 0 and all(r[1] in b.bridgeSet for r in b.relayLog)
assert all(b.squadOfTask[k] == a % 8 for k, xs in ch.posts.items() for *_, a, _ in xs)       # AC-3: nothing from a mirror
assert len(b.shareLog) == sum(len(xs) for xs in ch.posts.values())

In [ ]:
# test_bridge_prompts (FR-B1, FR-B2, P20-15, P20-16, NFR-1): the bridge sentences and lines appear only with bridge keys;
# SQUAD_RULE is byte-identical and the mirror and bridge rules open with its one exception; a guarding bridge keeps GUARD_RULE;
# only holders of the cross clue (and liar slots) see a mirror line; at most three new lines; pre-Phase-20 prompts, rows and
# numpy rows are byte-identical (hashes recorded before Phase 20's hooks)
import hashlib, json
from explRL.clue_colony import GUARD_RULE, SQUAD_RULE
from explRL.clue_runs import runClueNumpy
from explRL.colony_trace import replay
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply

assert SQUAD_RULE == ("Your team works in squads of {S}: you see, open and share only your squad's subtasks (about {n} of the {M} on "
                      "the board), and only your squad sees your posts.")
assert all(r.startswith('This is the one exception to the squad rule: ') for r in (MIRROR_RULE, BRIDGE_RULE))
heads = [h.split('{')[0] for h in (MIRROR_HEAD, PAIR_HEAD, NEED_HEAD)]
for bg in (False, True):
    b = BridgeBoardEnv(1800, 64, guardSlots=range(16, 32), squads=8, nLiar=16, couple=0.25, bridges=1, bridgeGuard=bg, **kw)
    env, ch = TextClueEnv(b, colony=True, guards=16, nLiar=16, squads=8), ClueBoard()
    assert env.slotRules and 'exception' not in env.rules()
    for i in range(16, 64):
        txt, s = env.rules(i), i % 8
        assert txt.startswith(TextClueEnv(ColonyBoardEnv(1800, 64, guardSlots=range(16, 32), squads=8, **kw), colony=True, guards=16,
                                          nLiar=16, squads=8).rules(i))           # the squads board's rules, then the bridge sentences
        assert COUPLE_RULE.format(s2=(s + 1) % 8) in txt and MIRROR_RULE.format(s0=(s - 1) % 8) in txt
        assert (BRIDGE_RULE.format(s2=(s + 1) % 8) in txt) == (i in b.bridgeSet) and (GUARD_RULE in txt) == (16 <= i < 32)
        assert txt.index(SQUAD_RULE.format(S=8, n=16, M=130)) < txt.index('This is the one exception')
        o = env.observe(i, 0, ch).split('\n')
        new = [x for x in o if any(x.startswith(h) for h in heads)]
        assert len(new) <= 3 and len({x[:12] for x in new}) == len(new)
        assert any(x.startswith(heads[0]) for x in new) == bool(b.mirrors(i)) == bool(squadView(b, i).crossClues())   # P20-16
        assert any(x.startswith(heads[1]) for x in new) == (i in b.bridgeSet)
        assert any(x.startswith(heads[2]) for x in new) == any(b.coupled.get(k, (None,))[0] == s for k in b.open)
    assert all(bridgeLines(squadView(b, i), i)[0].startswith(heads[0]) for i in range(16) if b.mirrors(i))   # liar slots: all mirrors
    assert sum(bool(b.mirrors(i)) for i in range(16, 64)) < 48                          # some honest agents see no mirror line
# no bridge key: no sentence or line, and bridgeLines / parseBridge / bridgeRules do nothing
c = ColonyBoardEnv(1800, 64, guardSlots=range(16, 32), squads=8, **kw)
e0 = TextClueEnv(c, colony=True, guards=16, nLiar=16, squads=8)
assert all('exception' not in e0.rules(i) and 'needs squad' not in e0.rules(i) for i in range(64))
assert not any(h in e0.observe(i, 0, ClueBoard()) for i in range(16, 64) for h in heads)
assert bridgeLines(squadView(c, 20), 20) == [] and parseBridge(e0, dict(action='share', target='T1', clue='x mod 3 = 1'), 20) is None
assert bridgeRules(e0, 20) == '' and bridgeRules(env, None) == ''
# NFR-1: Claude prompts, rows and numpy rows of pre-Phase-20 cells, hashed before the hooks went in
rnd = lambda x: {k: (round(v, 9) if isinstance(v, float) else v) for k, v in x.items()} if isinstance(x, dict) else [rnd(y) for y in x] if isinstance(x, list) else x
H = lambda x: hashlib.sha256(json.dumps(rnd(x), sort_keys=True, default=str).encode()).hexdigest()[:16]
PRE20 = [(8, 2, dict(m=2, r=1, R=30), ('40c44e8e1e454d46', 'f3469697e5ed8e1f', '71353a6be57585bc')),
         (16, 4, dict(kw, colony=True, guards=4, squads=4), ('4b9da269b2c3e769', '338865f1483024b8', '09babf683dad536c')),
         (16, 4, dict(kw, colony=True, guards=4, squads=8, failFlag=True), ('15b339edf30e756c', '1038bf32dea16a30', 'b5055174fadbb905')),
         (16, 4, dict(kw, colony=True, guards=4, squads=4, fault='adaptive'), ('7df9626a1e322530', '524b58d478e6d43b', '74234e98afe7498f'))]
for N, nL, w, want in PRE20:
    cfg = LLMConfig(task='clue', channel='blackboard', N=N, turns=6, nLiar=nL, memory='full', world=w)
    fc = FakeClient(fakeReply)
    rp = replay(cfg, 905, LLMClient('claude-haiku-5-5', client=fc))
    got = (H([(r['system'][0]['text'], r['messages']) for r in fc.requests]), H(rp.rows),
           H(runClueNumpy(cfg, 905) if w.get('fault') != 'adaptive' else None))
    assert got == want, (w, got)

In [ ]:
# test_rollout_bridges (AC-7, FR-B3, FR-B5, FR-B6): a 16-agent rollout in squads of 4 with one bridge per pair (fake policy: the
# fake client, with holders sharing their mirror clue and bridges relaying) runs on the bridge board, logs relays, refuses no
# prompt and, with the same replies, equals runClue's rows and turn log; bridgePromptTokens gives its longest prompt per role;
# the adaptive-liar path builds on a bridge board with the fake client, and its liars post on mirrors
import hashlib, json, re
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import coopRows, rollout
from explRL.clue_runs import runClue
from explRL.colony_agents import scriptedFaults
from explRL.colony_trace import replay
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply

def bridgeFake(req):
    "fakeReply, except that a bridge relays its pair's first mirror value and a holder shares its mirror clue on some turns."
    o = req['messages'][0]['content'][-1]['text']
    h = int(hashlib.sha256(o.encode()).hexdigest()[:8], 16)
    say = lambda k, c: json.dumps(dict(action='share', target=f'T{k}', answer=0, clue=c, notes=f'h{h % 7}'))
    if (m := re.search(r'^From squad \d+ for your subtasks: T(\d+): (x mod \d+ = \d+)', o, re.M)) and h % 2:
        return say(*m.groups())
    if (m := re.search(r'^For squad \d+ \(share your clue; you cannot answer\): .*?T(\d+): (x mod \d+ = \d+) \(yours\)', o, re.M)) and h % 3:
        return say(*m.groups())
    if 'secretly' in req['system'][0]['text'] and (m := re.search(r'^For squad .*?T(\d+): x mod (\d+)', o, re.M)) and h % 2:
        return say(m.group(1), f'x mod {m.group(2)} = {h % int(m.group(2))}')
    return fakeReply(req)

fake = lambda: LLMClient('claude-haiku-5-5', client=FakeClient(bridgeFake))
W16 = dict(kw, colony=True, guards=4, squads=4, couple=0.25, bridges=1)
cfg = LLMConfig(task='clue', channel='blackboard', N=16, turns=10, nLiar=4, memory='notes', world=W16)
ref, refLog = runClue(cfg, 901, fake())
ro = rollout(cfg, [901], claudePolicy(fake()), scriptedFaults(W16, 4), 'notes', keepPrompts=True)
bd, mine = ro.boards[0], sorted((x for lg in ro.logs[0] for x in lg), key=lambda x: (x['t'], x['agent']))
assert isinstance(bd, BridgeBoardEnv) and {k: v for k, v in coopRows(cfg, ro)[0].items() if k in ref[0]} == ref[0]
assert [dict(x, episode=0) for x in mine] == sorted(refLog, key=lambda x: (x['t'], x['agent']))
assert bd.relayLog and all(r[1] in bd.bridgeSet for r in bd.relayLog) and any(m[1] < 4 for m in bd.mirrorLog)   # scripted liars too
assert any(m[1] >= 4 and not m[6] for m in bd.mirrorLog) and all(x['refusal'] is None and x['stop'] != 'refusal' for x in mine)
relays = {(r[0], r[1]) for r in bd.relayLog}
assert all(x['report'] is not None for x in mine if (x['t'], x['agent']) in relays)                # relays reach the channel
assert all(x['report'] is None for x in mine if (x['t'], x['agent']) in {(m[0], m[1]) for m in bd.mirrorLog})   # mirror shares do not
sys_ = {i: ps[0][0] for i, ps in enumerate(ro.prompts[0]) if ps}
assert all((BRIDGE_RULE.format(s2=(i + 1) % 4) in s) == (i in bd.bridgeSet) for i, s in sys_.items())
# FR-B6: the longest prompt per role (bridge, mirror holder, other), here by the 4-characters-a-token estimate
L = bridgePromptTokens(cfg, [901, 902], policy=claudePolicy(fake()))
assert set(L) == {'bridge', 'holder', 'other'} and all(0 < v < 3072 for v in L.values()), L
assert bridgePromptTokens(cfg, [901], tok=len, policy=claudePolicy(fake()))['bridge'] > 3 * L['bridge']    # a text -> count tokenizer
print('longest prompt per role (about 4 characters a token):', L)
# FR-B5: adaptive (Claude) liars on a bridge board, with the fake client: they get the deceiver and bridge sentences and post on mirrors
cfgA = LLMConfig(task='clue', channel='blackboard', N=16, turns=10, nLiar=4, memory='notes', world=dict(W16, fault='adaptive'))
fc = FakeClient(bridgeFake)
rp = replay(cfgA, 903, LLMClient('claude-haiku-5-5', client=fc))
assert isinstance(rp.board, BridgeBoardEnv) and any(m[1] < 4 for m in rp.board.mirrorLog)
assert all(x['valid'] for x in rp.log if x['agent'] < 4 and x['report'] is None and x['action'][0] == 'share')
liarSys = [r['system'][0]['text'] for r in fc.requests if 'secretly' in r['system'][0]['text']]
assert liarSys and all('This is the one exception to the squad rule' in s for s in liarSys)

In [ ]:
# T2's recorded board: a 64-agent bridge1 board (C1's world: WORLD18, colony, 16 independent liars, 16 guards, squads of 8,
# one bridge per pair; the pilot seed 1890) with the fake client, replayed as runClue; its turn log shows mirror lines, bridge
# lines and the coupled-subtask line, and relays (Claude bridges' shares logged in relayLog with their source posts)
W64 = dict(kw, colony=True, guards=16, squads=8, couple=0.25, bridges=1)
cfg = LLMConfig(task='clue', channel='blackboard', N=64, turns=6, nLiar=16, memory='full', world=W64)
rp = replay(cfg, 1890, fake())
bd, log = rp.board, rp.log
assert len(log) == 48 * 6 and isinstance(bd, BridgeBoardEnv)
has = lambda h: [x for x in log if any(l.startswith(h) for l in x['obs'].split('\n'))]
mirrorTurns, pairTurns, needTurns = (has(h.split('{')[0]) for h in (MIRROR_HEAD, PAIR_HEAD, NEED_HEAD))
assert mirrorTurns and pairTurns and needTurns and {x['agent'] for x in pairTurns} == bd.bridgeSet
assert any(re.search(r'^From squad \d+ for your subtasks: T\d+: x mod', x['obs'], re.M) for x in pairTurns)   # posts reached a bridge
relayed = [x for x in log if (x['t'], x['agent']) in {(r[0], r[1]) for r in bd.relayLog}]
assert relayed and all(x['action'][0] == 'share' and x['agent'] in bd.bridgeSet for x in relayed)
assert all(r[5] in bd.mirrorLog for r in bd.relayLog) and any(m[1] >= 16 for m in bd.mirrorLog) and any(m[1] < 16 for m in bd.mirrorLog)
assert sum(not x['valid'] for x in log) / len(log) < 0.05
print(f"64 agents, 6 turns: {len(mirrorTurns)} turns with a mirror line, {len(pairTurns)} bridge turns, {len(bd.mirrorLog)} mirror "
      f"posts ({sum(m[6] for m in bd.mirrorLog)} false), {len(bd.relayLog)} relays ({sum(r[5][6] for r in bd.relayLog)} of false values)")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()